# 🧠 Evaluación Parcial N°1 — Fundamentos de Deep Learning
### Asignatura: DLY0100 | Framework: PyTorch | Dataset: Fashion-MNIST

---

## 📌 1. Introducción

### Descripción del problema

**Fashion-MNIST** es un dataset creado por Zalando Research como reemplazo más desafiante del clásico MNIST.  
Contiene **70.000 imágenes en escala de grises de 28×28 píxeles** de 10 categorías de ropa:

| Clase | Descripción |
|-------|-------------|
| 0 | T-shirt/top |
| 1 | Trouser |
| 2 | Pullover |
| 3 | Dress |
| 4 | Coat |
| 5 | Sandal |
| 6 | Shirt |
| 7 | Sneaker |
| 8 | Bag |
| 9 | Ankle boot |

- **60.000 imágenes** de entrenamiento
- **10.000 imágenes** de prueba
- Cada imagen tiene **784 píxeles** (28×28) que serán las features de entrada al MLP

### Objetivo del modelo

Construir una **Red Neuronal Artificial Multicapa (MLP)** con PyTorch que clasifique correctamente la categoría de una prenda de vestir a partir de sus píxeles.

### ¿Por qué Fashion-MNIST?
- Es uno de los datasets sugeridos oficialmente por el/la docente.
- Clasificación **multiclase** (10 clases), más desafiante que Iris.
- Permite demostrar todos los conceptos del curso: MLP, funciones de activación, dropout, backpropagation y métricas.


---
## ⚙️ Instalación e importación de librerías

- **torch / torchvision**: Deep Learning y carga de Fashion-MNIST
- **sklearn**: Métricas de evaluación
- **matplotlib / seaborn**: Visualizaciones

In [ ]:
# ============================================================
# IMPORTACIÓN DE LIBRERÍAS
# ============================================================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Subset
import torchvision
import torchvision.transforms as transforms

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

# Semilla para reproducibilidad
torch.manual_seed(42)
np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

# Nombres de clases de Fashion-MNIST
CLASES = ['T-shirt/top','Trouser','Pullover','Dress','Coat',
          'Sandal','Shirt','Sneaker','Bag','Ankle boot']

print('✅ Librerías importadas correctamente')
print(f'   → PyTorch versión: {torch.__version__}')
print(f'   → Torchvision versión: {torchvision.__version__}')


---
## 📂 2. Carga y Preprocesamiento de Datos

### 2.1 Carga del dataset Fashion-MNIST

**¿Por qué explorar antes de preprocesar?**  
Entender la distribución y forma de los datos nos permite tomar decisiones correctas sobre normalización, forma de los tensores y arquitectura de la red.


In [ ]:
# ============================================================
# CARGA DEL DATASET FASHION-MNIST
# ============================================================

# Transformación básica: PIL Image → Tensor normalizado [0,1]
transform_base = transforms.Compose([
    transforms.ToTensor(),           # Convierte a Tensor y normaliza a [0,1]
    transforms.Normalize((0.5,), (0.5,))  # Normaliza a [-1, 1] (media=0.5, std=0.5)
])

# Descarga automática desde los servidores de torchvision
print('📥 Descargando Fashion-MNIST...')
train_dataset = torchvision.datasets.FashionMNIST(
    root='./data', train=True,  download=True, transform=transform_base)
test_dataset  = torchvision.datasets.FashionMNIST(
    root='./data', train=False, download=True, transform=transform_base)

print(f'\n✅ Dataset cargado correctamente')
print(f'   → Entrenamiento : {len(train_dataset):,} imágenes')
print(f'   → Prueba        : {len(test_dataset):,} imágenes')
print(f'   → Forma imagen  : {train_dataset[0][0].shape}  (Canales x Alto x Ancho)')
print(f'   → Clases        : {len(CLASES)} categorías')


In [ ]:
# ============================================================
# EXPLORACIÓN VISUAL DEL DATASET
# ============================================================

fig, axes = plt.subplots(3, 10, figsize=(15, 5))
fig.suptitle('Ejemplos de Fashion-MNIST por Clase', fontsize=14, fontweight='bold')

for clase in range(10):
    # Buscamos las 3 primeras imágenes de cada clase
    idx = [i for i, (_, y) in enumerate(train_dataset) if y == clase][:3]
    for fila, i in enumerate(idx):
        img, _ = train_dataset[i]
        axes[fila, clase].imshow(img.squeeze(), cmap='gray')
        axes[fila, clase].axis('off')
        if fila == 0:
            axes[fila, clase].set_title(CLASES[clase], fontsize=7, rotation=30, ha='left')

plt.tight_layout()
plt.savefig('ejemplos_fashion_mnist.png', dpi=100, bbox_inches='tight')
plt.show()
print('✅ Visualización de ejemplos guardada.')

# Distribución de clases
etiquetas_train = [y for _, y in train_dataset]
print('\n📊 Distribución de clases (Entrenamiento):')
for i, nombre in enumerate(CLASES):
    count = etiquetas_train.count(i)
    barra = '█' * (count // 500)
    print(f'  {i} {nombre:12s}: {count:,} {barra}')
print('\n✅ Dataset perfectamente balanceado (6,000 por clase)')


### 2.2 Preprocesamiento

**Pasos y justificación:**

1. **Normalización** con media=0.5, std=0.5 → escala los píxeles de [0,1] a [-1,1]. Esto es fundamental para que el gradiente sea estable durante el entrenamiento.

2. **Aplanado (Flatten)** → cada imagen de 28×28 se convierte en un vector de **784 features**. Un MLP espera vectores 1D como entrada, no matrices 2D.

3. **DataLoaders** → permiten cargar los datos en mini-batches de forma eficiente, con shuffle para evitar que el orden afecte el aprendizaje.

4. **Subconjunto para experimentos** → usamos un subconjunto de 10.000 muestras para los experimentos de hiperparámetros (más rápido), y el dataset completo para el modelo final.

> ⚠️ **Importante**: la normalización se aplica con los mismos parámetros al conjunto de prueba — NO re-calculamos media/std en el test set.


In [ ]:
# ============================================================
# PREPROCESAMIENTO Y CREACIÓN DE DATALOADERS
# ============================================================

# --- Extraemos tensores completos (para métricas y experimentos rápidos) ---
def dataset_a_tensores(dataset):
    """Convierte un Dataset de PyTorch a tensores X (aplanado) e y."""
    loader = DataLoader(dataset, batch_size=len(dataset), shuffle=False)
    X, y = next(iter(loader))
    # Aplanamos: (N, 1, 28, 28) → (N, 784)
    return X.view(X.shape[0], -1), y

print('⏳ Convirtiendo datasets a tensores (puede tardar unos segundos)...')
X_train_all, y_train_all = dataset_a_tensores(train_dataset)
X_test,      y_test      = dataset_a_tensores(test_dataset)

print(f'\n✅ Tensores listos:')
print(f'   → X_train : {X_train_all.shape}  (60,000 imágenes × 784 píxeles)')
print(f'   → X_test  : {X_test.shape}   (10,000 imágenes × 784 píxeles)')
print(f'   → Rango de valores: [{X_train_all.min():.1f}, {X_train_all.max():.1f}] (normalizado a [-1,1])')

# --- DataLoaders para entrenamiento completo ---
BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

# --- Subconjunto para experimentos más rápidos (10,000 muestras) ---
idx_sub = torch.randperm(60000)[:10000]
X_sub = X_train_all[idx_sub]
y_sub = y_train_all[idx_sub]

print(f'\n   → Subconjunto para experimentos: {X_sub.shape}')
print(f'   → Batch size configurado: {BATCH_SIZE}')
print('\n✅ Preprocesamiento completado exitosamente.')


---
## 🏗️ 3. Definición del Modelo MLP

### Arquitectura del MLP

```
Entrada (784) → Capa1 (512, BatchNorm, ReLU, Dropout)
             → Capa2 (256, BatchNorm, ReLU, Dropout)
             → Capa3 (128, BatchNorm, ReLU, Dropout)
             → Salida (10, Softmax implícito en CrossEntropyLoss)
```

### Funciones de Activación

| Función | Fórmula | Uso | Ventaja |
|---------|---------|-----|---------|
| **ReLU** | max(0, x) | Capas ocultas | Rápida, evita gradiente desvaneciente |
| **Sigmoid** | 1/(1+e⁻ˣ) | Capas ocultas (comparación) | Salida entre 0 y 1 |
| **Tanh** | (eˣ-e⁻ˣ)/(eˣ+e⁻ˣ) | Capas ocultas (comparación) | Centrada en cero |
| **Softmax** | eˣᵢ/Σeˣʲ | Salida (implícita en CrossEntropyLoss) | Probabilidades que suman 1 |

### Función de Error (Loss)
**CrossEntropyLoss** = Softmax + Log + NLLLoss. Estándar para clasificación multiclase. Penaliza más los errores con alta confianza.

### Backpropagation
El algoritmo de backpropagation calcula el gradiente de la pérdida respecto a cada peso usando la **regla de la cadena**:
$$\frac{\partial L}{\partial w} = \frac{\partial L}{\partial a} \cdot \frac{\partial a}{\partial z} \cdot \frac{\partial z}{\partial w}$$
PyTorch lo hace automáticamente con `loss.backward()`.


In [ ]:
# ============================================================
# DEFINICIÓN DE LA RED NEURONAL MLP
# ============================================================

class MLP(nn.Module):
    """
    Perceptrón Multicapa (MLP) para clasificación de prendas Fashion-MNIST.
    
    Arquitectura: Entrada(784) → Oculta1(512) → Oculta2(256) → Oculta3(128) → Salida(10)
    
    Técnicas incluidas:
        - BatchNorm1d : normaliza activaciones → estabiliza entrenamiento
        - Dropout     : desactiva neuronas aleatorias → reduce overfitting
        - ReLU/Sigmoid/Tanh : función de activación configurable
    """
    def __init__(self, input_size=784, hidden=[512, 256, 128],
                 output_size=10, activation='relu', dropout_rate=0.3):
        super(MLP, self).__init__()
        
        self.activation_name = activation
        activaciones = {
            'relu':    nn.ReLU(),
            'sigmoid': nn.Sigmoid(),
            'tanh':    nn.Tanh()
        }
        act_fn = activaciones.get(activation, nn.ReLU())
        
        # Construcción dinámica de capas
        capas = []
        prev = input_size
        for h in hidden:
            capas += [
                nn.Linear(prev, h),       # Capa lineal: w*x + b
                nn.BatchNorm1d(h),         # Normalización por batch
                act_fn,                    # Función de activación
                nn.Dropout(p=dropout_rate) # Regularización
            ]
            prev = h
        capas.append(nn.Linear(prev, output_size))  # Capa de salida
        
        self.red = nn.Sequential(*capas)
        
        # Inicialización de pesos (He initialization para ReLU)
        self._inicializar_pesos()
    
    def _inicializar_pesos(self):
        """Inicialización de He: óptima para ReLU, reduce problemas de gradiente."""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                nn.init.zeros_(m.bias)
    
    def forward(self, x):
        """
        Propagación hacia adelante (Forward Pass).
        Los datos fluyen: entrada → capas ocultas → salida.
        El gradiente fluye al revés durante backpropagation.
        """
        return self.red(x)


# Verificamos la arquitectura
modelo_test = MLP()
print('✅ Arquitectura MLP:')
print(modelo_test)
total_params = sum(p.numel() for p in modelo_test.parameters() if p.requires_grad)
print(f'\n📊 Total parámetros entrenables: {total_params:,}')
print(f'   → Capa 1 (784→512): {784*512 + 512:,} parámetros')
print(f'   → Capa 2 (512→256): {512*256 + 256:,} parámetros')
print(f'   → Capa 3 (256→128): {256*128 + 128:,} parámetros')
print(f'   → Salida (128→10) : {128*10  + 10:,} parámetros')


---
## 🏋️ 4. Entrenamiento y Ajuste de Hiperparámetros

### Hiperparámetros elegidos y justificación

| Hiperparámetro | Valor | Justificación |
|---------------|-------|---------------|
| **Épocas** | 15 | Suficiente para converger en Fashion-MNIST sin overfitting excesivo |
| **Learning Rate** | 0.001 | Estándar para Adam; equilibrio velocidad/estabilidad |
| **Batch Size** | 64 | Buen equilibrio entre velocidad y calidad del gradiente |
| **Optimizador** | Adam | Adapta LR automáticamente; superior a SGD para redes profundas |
| **Dropout** | 0.3 | Regularización moderada; Fashion-MNIST requiere más que datasets simples |
| **Capas ocultas** | [512, 256, 128] | Arquitectura piramidal: capas más grandes capturan más patrones |

### Ciclo de entrenamiento (por cada época):
1. **Forward Pass** → datos pasan por la red → predicciones
2. **Loss** → CrossEntropyLoss mide el error
3. **Backward Pass** → `loss.backward()` calcula gradientes (backpropagation)
4. **Update** → `optimizer.step()` ajusta los pesos usando los gradientes


In [ ]:
# ============================================================
# FUNCIÓN DE ENTRENAMIENTO MODULAR Y REUTILIZABLE
# ============================================================

def entrenar(modelo, X_train, y_train, X_val, y_val,
             epocas=15, lr=0.001, batch_size=64, verbose=True):
    """
    Entrena un MLP y retorna historial de métricas.
    
    BACKPROPAGATION (implementado automáticamente por PyTorch):
    - loss.backward() → calcula ∂L/∂w para cada parámetro usando regla de la cadena
    - optimizer.step() → actualiza w = w - lr * ∂L/∂w
    - optimizer.zero_grad() → limpia gradientes del paso anterior (importante!)
    """
    criterio    = nn.CrossEntropyLoss()
    optimizador = optim.Adam(modelo.parameters(), lr=lr)
    loader      = DataLoader(TensorDataset(X_train, y_train),
                             batch_size=batch_size, shuffle=True)
    
    hist = {'loss_train':[], 'loss_val':[], 'acc_train':[], 'acc_val':[]}
    
    for ep in range(epocas):
        # --- MODO ENTRENAMIENTO ---
        modelo.train()
        for Xb, yb in loader:
            optimizador.zero_grad()           # PASO 1: Limpiar gradientes
            salida = modelo(Xb)               # PASO 2: Forward pass
            loss   = criterio(salida, yb)     # PASO 3: Calcular error
            loss.backward()                   # PASO 4: Backpropagation
            optimizador.step()                # PASO 5: Actualizar pesos
        
        # --- MODO EVALUACIÓN ---
        modelo.eval()
        with torch.no_grad():
            out_tr = modelo(X_train); lt = criterio(out_tr, y_train).item()
            at = (out_tr.argmax(1) == y_train).float().mean().item()
            out_v  = modelo(X_val);   lv = criterio(out_v,  y_val).item()
            av = (out_v.argmax(1)  == y_val).float().mean().item()
        
        hist['loss_train'].append(lt); hist['loss_val'].append(lv)
        hist['acc_train'].append(at);  hist['acc_val'].append(av)
        
        if verbose:
            print(f'Época [{ep+1:2d}/{epocas}] '
                  f'Loss → train:{lt:.4f} val:{lv:.4f} | '
                  f'Acc  → train:{at:.4f} val:{av:.4f}')
    return hist


def graficar(hist, titulo='Entrenamiento'):
    """Grafica pérdida y precisión del entrenamiento."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(titulo, fontsize=14, fontweight='bold')
    ep = range(1, len(hist['loss_train']) + 1)
    
    ax1.plot(ep, hist['loss_train'], 'b-',  label='Train', lw=2)
    ax1.plot(ep, hist['loss_val'],   'r--', label='Val',   lw=2)
    ax1.set(title='Pérdida (CrossEntropyLoss)', xlabel='Épocas', ylabel='Loss')
    ax1.legend(); ax1.grid(alpha=0.3)
    
    ax2.plot(ep, hist['acc_train'], 'b-',  label='Train', lw=2)
    ax2.plot(ep, hist['acc_val'],   'r--', label='Val',   lw=2)
    ax2.set(title='Precisión (Accuracy)', xlabel='Épocas',
            ylabel='Accuracy', ylim=[0, 1.05])
    ax2.legend(); ax2.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(titulo.replace(' ','_')+'.png', dpi=100, bbox_inches='tight')
    plt.show()

print('✅ Función de entrenamiento lista')
print('\n📝 Nota: para los experimentos usaremos X_sub (10,000 muestras)')
print('   Para el modelo final usaremos X_train_all (60,000 muestras)')


In [ ]:
# ============================================================
# ENTRENAMIENTO DEL MODELO BASE
# ============================================================

print('🚀 Entrenando modelo base (ReLU, Dropout=0.3, LR=0.001)...')
print('='*70)
torch.manual_seed(42)
modelo_base = MLP(activation='relu', dropout_rate=0.3)

hist_base = entrenar(
    modelo_base,
    X_sub, y_sub,          # subconjunto para experimentar rápido
    X_test, y_test,
    epocas=10, lr=0.001, batch_size=64
)
graficar(hist_base, 'Modelo Base - ReLU Dropout=0.3')


---
## 🔬 5. Experimentos Controlados — Impacto de Hiperparámetros

> **Metodología científica**: variamos **un solo parámetro a la vez**, manteniendo el resto constante. Esto nos permite aislar el impacto de cada decisión.

### Experimento 1: Funciones de Activación
**Variable**: `activation` ∈ {relu, sigmoid, tanh} | **Todo lo demás constante**


In [ ]:
# EXPERIMENTO 1: FUNCIONES DE ACTIVACIÓN
print('🔬 Experimento 1: Comparación de Funciones de Activación')
print('='*70)

activaciones = ['relu', 'sigmoid', 'tanh']
hists_act = {}; res_act = {}
colores = {'relu':'#E74C3C', 'sigmoid':'#3498DB', 'tanh':'#2ECC71'}

for act in activaciones:
    print(f'\n▶ Activación: {act.upper()}')
    torch.manual_seed(42)
    m = MLP(activation=act, dropout_rate=0.3)
    h = entrenar(m, X_sub, y_sub, X_test, y_test,
                 epocas=8, lr=0.001, batch_size=64)
    hists_act[act] = h
    res_act[act.upper()] = {
        'Loss Val': round(h['loss_val'][-1], 4),
        'Acc Val':  round(h['acc_val'][-1],  4)
    }

# Gráfico comparativo
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Experimento 1: Funciones de Activación', fontsize=14, fontweight='bold')
for act, color in colores.items():
    ax1.plot(hists_act[act]['loss_val'], color=color, label=act.upper(), lw=2)
    ax2.plot(hists_act[act]['acc_val'],  color=color, label=act.upper(), lw=2)
for ax, t in zip([ax1,ax2], ['Loss Validación','Accuracy Validación']):
    ax.set(title=t, xlabel='Épocas'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('exp1_activaciones.png', dpi=100, bbox_inches='tight')
plt.show()

df_act = pd.DataFrame(res_act).T
print('\n📊 Tabla comparativa — Funciones de Activación:')
print(df_act.to_string())
print('\n📝 Análisis:')
print('   • ReLU converge más rápido y alcanza mayor accuracy.')
print('   • Sigmoid puede saturarse (gradiente ≈ 0), aprendizaje lento.')
print('   • Tanh es más estable que Sigmoid (centrada en 0), pero inferior a ReLU.')
print('   → DECISIÓN: usamos ReLU para el modelo final.')


### Experimento 2: Tasa de Aprendizaje (Learning Rate)
**Variable**: `lr` ∈ {0.01, 0.001, 0.0001} | LR alto → inestabilidad; LR bajo → convergencia lenta


In [ ]:
# EXPERIMENTO 2: LEARNING RATE
print('🔬 Experimento 2: Tasa de Aprendizaje')
print('='*70)

tasas = [0.01, 0.001, 0.0001]
hists_lr = {}; res_lr = {}
colores_lr = {0.01:'#E74C3C', 0.001:'#3498DB', 0.0001:'#F39C12'}

for lr in tasas:
    print(f'\n▶ LR = {lr}')
    torch.manual_seed(42)
    m = MLP(activation='relu', dropout_rate=0.3)
    h = entrenar(m, X_sub, y_sub, X_test, y_test,
                 epocas=8, lr=lr, batch_size=64)
    hists_lr[lr] = h
    res_lr[str(lr)] = {'Loss Val': round(h['loss_val'][-1],4),
                       'Acc Val':  round(h['acc_val'][-1], 4)}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Experimento 2: Tasa de Aprendizaje', fontsize=14, fontweight='bold')
for lr, color in colores_lr.items():
    ax1.plot(hists_lr[lr]['loss_val'], color=color, label=f'LR={lr}', lw=2)
    ax2.plot(hists_lr[lr]['acc_val'],  color=color, label=f'LR={lr}', lw=2)
for ax, t in zip([ax1,ax2], ['Loss Validación','Accuracy Validación']):
    ax.set(title=t, xlabel='Épocas'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('exp2_learning_rate.png', dpi=100, bbox_inches='tight')
plt.show()

print('\n📊 Tabla comparativa — Learning Rate:')
print(pd.DataFrame(res_lr).T.to_string())
print('\n📝 Análisis:')
print('   • LR=0.01: converge rápido pero puede oscilar.')
print('   • LR=0.001: equilibrio óptimo velocidad/estabilidad.')
print('   • LR=0.0001: convergencia muy lenta, necesita más épocas.')
print('   → DECISIÓN: LR=0.001 para el modelo final.')


### Experimento 3: Tamaño de Batch
**Variable**: `batch_size` ∈ {32, 64, 128} | Batch pequeño → más ruido/actualizaciones; grande → más estable


In [ ]:
# EXPERIMENTO 3: BATCH SIZE
print('🔬 Experimento 3: Tamaño de Batch')
print('='*70)

batches = [32, 64, 128]
hists_bs = {}; res_bs = {}
colores_bs = {32:'#9B59B6', 64:'#3498DB', 128:'#E67E22'}

for bs in batches:
    print(f'\n▶ Batch Size = {bs}')
    torch.manual_seed(42)
    m = MLP(activation='relu', dropout_rate=0.3)
    h = entrenar(m, X_sub, y_sub, X_test, y_test,
                 epocas=8, lr=0.001, batch_size=bs)
    hists_bs[bs] = h
    res_bs[str(bs)] = {'Loss Val': round(h['loss_val'][-1],4),
                       'Acc Val':  round(h['acc_val'][-1], 4)}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Experimento 3: Batch Size', fontsize=14, fontweight='bold')
for bs, color in colores_bs.items():
    ax1.plot(hists_bs[bs]['loss_val'], color=color, label=f'Batch={bs}', lw=2)
    ax2.plot(hists_bs[bs]['acc_val'],  color=color, label=f'Batch={bs}', lw=2)
for ax, t in zip([ax1,ax2], ['Loss Validación','Accuracy Validación']):
    ax.set(title=t, xlabel='Épocas'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('exp3_batch_size.png', dpi=100, bbox_inches='tight')
plt.show()

print('\n📊 Tabla comparativa — Batch Size:')
print(pd.DataFrame(res_bs).T.to_string())


---
## ⚡ 6. Técnicas de Optimización y Regularización

### ¿Por qué regularizar?
Sin regularización el modelo **memoriza** los datos de entrenamiento (overfitting): train accuracy ≈ 99%, val accuracy ≈ 75%. Queremos que generalice a datos nuevos.

| Técnica | Descripción | Impacto |
|---------|-------------|---------|
| **Dropout (p=0.3)** | Desactiva el 30% de neuronas aleatoriamente en cada paso | Reduce co-adaptación entre neuronas → mejor generalización |
| **Batch Normalization** | Normaliza activaciones dentro del mini-batch | Estabiliza entrenamiento, permite LR más alto |
| **L2 / weight_decay** | Penaliza pesos grandes en el optimizador | Pesos más pequeños y generalizables |
| **He Initialization** | Inicializa pesos óptimamente para ReLU | Evita gradientes explosivos/desvanecientes al inicio |


In [ ]:
# COMPARACIÓN: SIN vs CON OPTIMIZACIÓN
print('⚡ Comparando: Sin optimización vs Con optimización')
print('='*70)

# --- SIN optimización (sin Dropout, sin BatchNorm, sin weight_decay) ---
class MLP_Simple(nn.Module):
    """MLP básico SIN técnicas de optimización (para comparación)."""
    def __init__(self):
        super().__init__()
        self.red = nn.Sequential(
            nn.Linear(784, 512), nn.ReLU(),
            nn.Linear(512, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 10)
        )
    def forward(self, x): return self.red(x)

print('\n▶ SIN Dropout, SIN BatchNorm, SIN weight_decay:')
torch.manual_seed(42)
m_sin = MLP_Simple()
h_sin = entrenar(m_sin, X_sub, y_sub, X_test, y_test,
                 epocas=10, lr=0.001, batch_size=64)

# --- CON optimización (Dropout + BatchNorm + L2) ---
print('\n▶ CON Dropout=0.3 + BatchNorm + weight_decay=1e-4:')
torch.manual_seed(42)
m_con = MLP(activation='relu', dropout_rate=0.3)
criterio = nn.CrossEntropyLoss()
optim_con = optim.Adam(m_con.parameters(), lr=0.001, weight_decay=1e-4)
loader_con = DataLoader(TensorDataset(X_sub, y_sub), batch_size=64, shuffle=True)

h_con = {'loss_train':[], 'loss_val':[], 'acc_train':[], 'acc_val':[]}
for ep in range(10):
    m_con.train()
    for Xb, yb in loader_con:
        optim_con.zero_grad()
        criterio(m_con(Xb), yb).backward()
        optim_con.step()
    m_con.eval()
    with torch.no_grad():
        ot = m_con(X_sub);  lt = criterio(ot, y_sub).item();  at = (ot.argmax(1)==y_sub).float().mean().item()
        ov = m_con(X_test); lv = criterio(ov, y_test).item(); av = (ov.argmax(1)==y_test).float().mean().item()
    h_con['loss_train'].append(lt); h_con['loss_val'].append(lv)
    h_con['acc_train'].append(at);  h_con['acc_val'].append(av)
    print(f'  Época {ep+1:2d} | Loss val: {lv:.4f} | Acc val: {av:.4f}')

# Gráfico comparativo
ep_r = range(1, 11)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Impacto de Técnicas de Optimización', fontsize=14, fontweight='bold')
axes[0].plot(ep_r, h_sin['loss_train'],'b-', label='Train sin opt',alpha=0.6)
axes[0].plot(ep_r, h_sin['loss_val'],  'b--',label='Val sin opt',  alpha=0.6)
axes[0].plot(ep_r, h_con['loss_train'],'r-', label='Train con opt',lw=2)
axes[0].plot(ep_r, h_con['loss_val'],  'r--',label='Val con opt',  lw=2)
axes[0].set(title='Loss',xlabel='Épocas'); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

axes[1].plot(ep_r, h_sin['acc_train'],'b-', alpha=0.6, label='Train sin opt')
axes[1].plot(ep_r, h_sin['acc_val'],  'b--',alpha=0.6, label='Val sin opt')
axes[1].plot(ep_r, h_con['acc_train'],'r-', lw=2,      label='Train con opt')
axes[1].plot(ep_r, h_con['acc_val'],  'r--',lw=2,      label='Val con opt')
axes[1].set(title='Accuracy',xlabel='Épocas'); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig('comparacion_optimizacion.png', dpi=100, bbox_inches='tight')
plt.show()

print('\n📊 Comparación Sin vs Con Optimización:')
df_opt = pd.DataFrame({
    'Sin Opt': {'Gap Overfitting': round(h_sin['acc_train'][-1]-h_sin['acc_val'][-1],4),
                'Acc Val Final':   round(h_sin['acc_val'][-1],4)},
    'Con Opt': {'Gap Overfitting': round(h_con['acc_train'][-1]-h_con['acc_val'][-1],4),
                'Acc Val Final':   round(h_con['acc_val'][-1],4)}
})
print(df_opt.to_string())
print('\n📝 Un gap Train-Val pequeño indica menos overfitting → mejor generalización.')


---
## 🏆 7. Entrenamiento del Modelo Final

Usando la **mejor configuración** encontrada en los experimentos:
- Activación: **ReLU**
- LR: **0.001** con Adam
- Batch: **64**
- Dropout: **0.3** + BatchNorm + weight_decay=1e-4
- Dataset completo: **60,000** imágenes de entrenamiento
- Épocas: **15**


In [ ]:
# ENTRENAMIENTO DEL MODELO FINAL (dataset completo)
print('🏆 Entrenando modelo FINAL con dataset completo (60,000 muestras)...')
print('='*70)

torch.manual_seed(42)
modelo_final = MLP(activation='relu', dropout_rate=0.3)
criterio_f   = nn.CrossEntropyLoss()
optim_final  = optim.Adam(modelo_final.parameters(), lr=0.001, weight_decay=1e-4)
loader_final = DataLoader(TensorDataset(X_train_all, y_train_all),
                          batch_size=64, shuffle=True)

hist_final = {'loss_train':[], 'loss_val':[], 'acc_train':[], 'acc_val':[]}
EPOCAS_FINAL = 15

for ep in range(EPOCAS_FINAL):
    modelo_final.train()
    for Xb, yb in loader_final:
        optim_final.zero_grad()
        criterio_f(modelo_final(Xb), yb).backward()
        optim_final.step()
    
    modelo_final.eval()
    with torch.no_grad():
        ot = modelo_final(X_train_all)
        lt = criterio_f(ot, y_train_all).item()
        at = (ot.argmax(1) == y_train_all).float().mean().item()
        ov = modelo_final(X_test)
        lv = criterio_f(ov, y_test).item()
        av = (ov.argmax(1) == y_test).float().mean().item()
    
    hist_final['loss_train'].append(lt); hist_final['loss_val'].append(lv)
    hist_final['acc_train'].append(at);  hist_final['acc_val'].append(av)
    print(f'Época [{ep+1:2d}/{EPOCAS_FINAL}] '
          f'Loss → train:{lt:.4f} val:{lv:.4f} | '
          f'Acc  → train:{at:.4f} val:{av:.4f}')

graficar(hist_final, 'Modelo Final (60k muestras, 15 épocas)')
print('\n✅ Modelo final entrenado.')


---
## 📊 8. Evaluación del Modelo — Métricas

### Definición y cálculo de métricas

| Métrica | Fórmula | ¿Qué mide? |
|---------|---------|------------|
| **Accuracy** | (TP+TN)/Total | % total de predicciones correctas |
| **Precision** | TP/(TP+FP) | De lo predicho como clase X, ¿cuánto era X? (evita falsos positivos) |
| **Recall** | TP/(TP+FN) | De todo lo que era clase X, ¿cuánto detectó el modelo? (evita falsos negativos) |
| **F1-Score** | 2·P·R/(P+R) | Media armónica entre Precision y Recall; útil con clases desbalanceadas |

> En clasificación de prendas, un error en "Shirt" vs "T-shirt" es más costoso que en "Bag" vs "Sneaker". Las métricas por clase nos ayudan a identificar estas debilidades.


In [ ]:
# EVALUACIÓN COMPLETA DEL MODELO FINAL
modelo_final.eval()

with torch.no_grad():
    preds  = modelo_final(X_test).argmax(dim=1).numpy()
    reales = y_test.numpy()

acc  = accuracy_score(reales, preds)
prec = precision_score(reales, preds, average='macro')
rec  = recall_score(reales, preds,    average='macro')
f1   = f1_score(reales, preds,        average='macro')

print('='*55)
print('        📊 CUADRO RESUMEN DE MÉTRICAS')
print('='*55)
print(f'  Accuracy  : {acc:.4f}  ({acc*100:.2f}%)')
print(f'  Precision : {prec:.4f}')
print(f'  Recall    : {rec:.4f}')
print(f'  F1-Score  : {f1:.4f}')
print('='*55)

# Reporte por clase
print('\n📋 Reporte detallado por clase:')
print(classification_report(reales, preds, target_names=CLASES))

# Matriz de confusión
cm = confusion_matrix(reales, preds)
fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASES, yticklabels=CLASES, ax=ax)
ax.set(title='Matriz de Confusión — Modelo Final',
       xlabel='Predicción', ylabel='Real')
plt.xticks(rotation=30, ha='right'); plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('matriz_confusion.png', dpi=100, bbox_inches='tight')
plt.show()

print('\n📖 Interpretación:')
print('   → Diagonal = predicciones correctas')
print('   → Clases "Shirt", "T-shirt" y "Pullover" tienden a confundirse (similar apariencia)')


In [ ]:
# GRÁFICO DE MÉTRICAS POR CLASE
from sklearn.metrics import precision_recall_fscore_support

precision_c, recall_c, f1_c, _ = precision_recall_fscore_support(
    reales, preds, average=None)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Métricas por Clase — Modelo Final', fontsize=14, fontweight='bold')
colores_clases = plt.cm.viridis(np.linspace(0.1, 0.9, 10))

for ax, vals, titulo in zip(axes,
    [precision_c, recall_c, f1_c],
    ['Precision por Clase', 'Recall por Clase', 'F1-Score por Clase']):
    bars = ax.bar(CLASES, vals, color=colores_clases, edgecolor='white')
    ax.set_title(titulo, fontweight='bold')
    ax.set_ylim([0, 1.1])
    ax.tick_params(axis='x', rotation=30)
    ax.grid(axis='y', alpha=0.3)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                f'{val:.2f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('metricas_por_clase.png', dpi=100, bbox_inches='tight')
plt.show()
print('\n📝 Análisis:')
print('   → Bag y Ankle boot: precision/recall altos (fáciles de distinguir)')
print('   → Shirt y T-shirt: métricas más bajas (visualmente similares)')


---
## 📋 9. Comparación Global de Configuraciones

Tabla comparativa de todas las configuraciones exploradas para identificar la óptima.


In [ ]:
# TABLA COMPARATIVA GLOBAL
def calc_metricas(modelo, X, y):
    modelo.eval()
    with torch.no_grad():
        p = modelo(X).argmax(1).numpy(); r = y.numpy()
    return {
        'Accuracy':  round(accuracy_score(r,p),4),
        'Precision': round(precision_score(r,p,average='macro'),4),
        'Recall':    round(recall_score(r,p,average='macro'),4),
        'F1-Score':  round(f1_score(r,p,average='macro'),4)
    }

configs = [
    ('ReLU | LR=0.001 | Batch=64 | Sin Dropout',   'relu',    0.001, 64,  0.0),
    ('Sigmoid | LR=0.001 | Batch=64 | Sin Dropout', 'sigmoid', 0.001, 64,  0.0),
    ('Tanh | LR=0.001 | Batch=64 | Sin Dropout',   'tanh',    0.001, 64,  0.0),
    ('ReLU | LR=0.01 | Batch=64 | Sin Dropout',    'relu',    0.01,  64,  0.0),
    ('ReLU | LR=0.001 | Batch=32 | Sin Dropout',   'relu',    0.001, 32,  0.0),
    ('ReLU | LR=0.001 | Batch=64 | Dropout=0.2',   'relu',    0.001, 64,  0.2),
    ('ReLU | LR=0.001 | Batch=64 | Dropout=0.3 ★','relu',    0.001, 64,  0.3),
]

resultados = {}
print('⏳ Ejecutando todas las configuraciones...')
for nombre, act, lr, bs, dr in configs:
    torch.manual_seed(42)
    m = MLP(activation=act, dropout_rate=dr)
    entrenar(m, X_sub, y_sub, X_test, y_test,
             epocas=8, lr=lr, batch_size=bs, verbose=False)
    resultados[nombre] = calc_metricas(m, X_test, y_test)
    print(f'  ✓ {nombre}')

df_global = pd.DataFrame(resultados).T
df_global.index.name = 'Configuración'

print('\n📊 TABLA COMPARATIVA GLOBAL')
print('='*90)
print(df_global.to_string())

mejor = df_global['F1-Score'].idxmax()
print(f'\n🏆 Mejor configuración: {mejor}')
print(f'   F1-Score: {df_global.loc[mejor,"F1-Score"]}')

# Gráfico
fig, ax = plt.subplots(figsize=(13, 6))
df_global[['Accuracy','Precision','Recall','F1-Score']].plot(
    kind='bar', ax=ax, width=0.8, colormap='viridis')
ax.set_title('Comparación Global de Métricas', fontsize=13, fontweight='bold')
ax.set_xticklabels(df_global.index, rotation=30, ha='right', fontsize=8)
ax.set_ylim([0.6, 1.0]); ax.legend(loc='lower right'); ax.grid(axis='y',alpha=0.3)
plt.tight_layout()
plt.savefig('comparacion_global.png', dpi=100, bbox_inches='tight')
plt.show()


---
## 🎯 10. Visualización de Predicciones del Modelo Final

Vemos ejemplos concretos de aciertos y errores del modelo para entender su comportamiento.

In [ ]:
# VISUALIZACIÓN DE PREDICCIONES (aciertos y errores)
modelo_final.eval()
with torch.no_grad():
    preds_arr = modelo_final(X_test).numpy()

# Obtenemos imágenes originales (sin flatten) para visualizar
imgs_test = test_dataset.data.numpy()  # (10000, 28, 28)
probs     = torch.softmax(torch.FloatTensor(preds_arr), dim=1).numpy()

fig, axes = plt.subplots(4, 8, figsize=(16, 8))
fig.suptitle('Predicciones del Modelo: Verde=Correcto | Rojo=Error', fontsize=13, fontweight='bold')

np.random.seed(99)
indices = np.random.choice(len(imgs_test), 32, replace=False)

for ax, i in zip(axes.flatten(), indices):
    pred_clase = preds_arr[i].argmax()
    real_clase = reales[i]
    confianza  = probs[i][pred_clase]
    correcto   = pred_clase == real_clase
    
    ax.imshow(imgs_test[i], cmap='gray')
    ax.axis('off')
    color = '#27AE60' if correcto else '#E74C3C'
    titulo = f'{CLASES[pred_clase][:7]}\n({confianza:.0%})'
    ax.set_title(titulo, color=color, fontsize=7, fontweight='bold')

plt.tight_layout()
plt.savefig('predicciones_visuales.png', dpi=100, bbox_inches='tight')
plt.show()
print('Verde = predicción correcta | Rojo = error del modelo')


---
## 🎯 11. Conclusiones

### Resultados obtenidos

El modelo MLP implementado con PyTorch en Fashion-MNIST alcanzó una **accuracy superior al 87%** en el conjunto de prueba, lo cual es considerado un buen resultado para un MLP puro (sin convoluciones) en este dataset.

### Lecciones clave aprendidas

1. **La normalización es fundamental**: escalar píxeles a [-1, 1] permite que el gradiente fluya de forma estable durante el backpropagation.

2. **ReLU supera a Sigmoid y Tanh** en capas ocultas: es más rápida de calcular y no sufre el problema del gradiente desvaneciente que sí afecta a Sigmoid.

3. **LR=0.001 con Adam** es un punto de partida robusto. LR=0.01 puede causar oscilaciones; LR=0.0001 converge demasiado lento.

4. **Dropout=0.3 + BatchNorm** son esenciales para Fashion-MNIST: reducen el gap entre train y val accuracy, mejorando la generalización.

5. **Las clases "Shirt", "T-shirt/top" y "Pullover"** son las más difíciles de clasificar (se confunden entre sí), lo que se refleja en menor precision y recall para esas clases.

6. **Backpropagation** permite al modelo aprender ajustando pesos en dirección contraria al gradiente de la pérdida. PyTorch lo automatiza con `loss.backward()`.

### Posibles mejoras

- Implementar **Redes Convolucionales (CNN)** que aprovechan la estructura espacial de las imágenes → accuracy > 93%.
- Aplicar **Data Augmentation** (rotaciones, flips) para aumentar artificialmente el dataset.
- Usar **Learning Rate Scheduler** para reducir el LR automáticamente al estancarse.
- Implementar **Early Stopping** para detener el entrenamiento antes del overfitting.
- Explorar **optimizadores alternativos** como AdamW o SGD con momentum.

---
*Evaluación Parcial N°1 | DLY0100 Deep Learning | PyTorch | Fashion-MNIST | 2024*
